# Създаване на приложения за генериране на изображения (OpenAI)

Големите езикови модели не са само за генериране на текст. Можете също да генерирате изображения от текстови описания. Изображенията като модалност са полезни в медицинската техника, архитектурата, туризма, разработката на игри, маркетинга и други. В този урок работим с днешните **GPT Image** модели на платформата OpenAI.

## Учебни цели

В края на този урок ще можете да:

- Обясните какво е генериране на изображения и къде е полезно.
- Разберете семейството модели `gpt-image` и как се различават от старите модели DALL·E.
- Създадете приложение за генериране на изображения и редактирате изображения.

## Какво е генериране на изображения?

Моделите за генериране на изображения създават картини от текстови подканващи фрази. Съвременните модели като `gpt-image` изучават връзката между текст и изображения по време на обучението си и след това стъпково преобразуват случаен шум в изображение, което съответства на вашата подканваща фраза.

Две добре познати семейства модели за изображения са:

- **`gpt-image` (OpenAI)** — настоящото поколение, използвано в този урок. Поддържа генериране на изображения от текст и редактиране на изображения (поправяне с маска).
- **Midjourney** — популярен модел на трета страна с собствена услуга и работен процес в Discord.

> По-старите модели за изображения на OpenAI — **DALL·E 2** и **DALL·E 3** — са наследствени. DALL·E 3 вече не е достъпен за нови внедрявания, а функции като `create_variation` съществуваха само в DALL·E 2. Използвайте моделите `gpt-image` за нови приложения.

> **Важно:** Моделите `gpt-image` връщат генерираното изображение като **base64** (`b64_json`), а не като URL. Вашият код декодира base64 стринга в байтове и го записва — няма URL за изтегляне на изображение.


## Създаване на ваше първо приложение за генериране на изображения

Значи какво е нужно, за да създадете приложение за генериране на изображения? Трябва ви следните библиотеки:

- **python-dotenv**, силно се препоръчва да използвате тази библиотека, за да съхранявате тайните си в *.env* файл, отделен от кода.
- **openai**, тази библиотека ще използвате за взаимодействие с OpenAI API.
- **pillow**, за работа с изображения в Python.


1. Създайте файл *.env* със следното съдържание:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Съберете горните библиотеки в файл, наречен *requirements.txt*, по следния начин:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. След това създайте виртуална среда и инсталирайте библиотеките:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> За Windows използвайте следните команди, за да създадете и активирате вашата виртуална среда:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Добавете следния код във файл, наречен *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Създайте обект OpenAI (чете OPENAI_API_KEY от вашия .env)
    client = openai.OpenAI()


    try:
        # Създайте изображение, като използвате API за генериране на изображения
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Въведете текста на вашето подканване тук
            size='1024x1024',
            n=1
        )
        # Задайте директорията за съхранение на изображението
        image_dir = os.path.join(os.curdir, 'images')

        # Ако директорията не съществува, създайте я
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Инициализирайте пътя към изображението (забележете, че типът на файла трябва да е png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image моделите връщат изображението като base64 (b64_json), а не URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Покажете изображението в стандартния визуализатор на изображения
        image = Image.open(image_path)
        image.show()

    # обработва изключенията
    except openai.BadRequestError as err:
        print(err)

    ```

Нека обясним този код:

- Първо, импортираме нужните библиотеки, включително библиотеката OpenAI, библиотеката dotenv, модула base64 и библиотеката Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- След това създаваме клиента, който чете API ключа от вашия ``.env``.

    ```python
    # Създаване на OpenAI обект
    client = openai.OpenAI()
    ```

- След това генерираме изображението:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Въведете вашия текст тук
        size='1024x1024',
        n=1
    )
    ```

    Моделите `gpt-image` връщат изображението като **base64** низ в `data[0].b64_json`. Декодираме го до байтове и го записваме във файл — няма URL за изтегляне.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Накрая отваряме изображението и използваме стандартния визуализатор за изображения, за да го покажем:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Повече детайли за генерирането на изображението

Нека разгледаме параметрите на `images.generate`:

- **model**, е моделът за изображение, който ще използваме (например `gpt-image-1`).
- **prompt**, е текстовата подсказка, използвана за генериране на изображението. Тук е "Зайче на кон, държащо близалка, на мъглива ливада, където растат нарциси".
- **size**, е размерът на генерираното изображение (например `1024x1024`, `1536x1024`, `1024x1536` или `"auto"`).
- **n**, е броят на генерираните изображения. Тук генерираме едно.

> Моделите за изображения не използват параметъра `temperature` — той е за контрол на генерацията на текст. За да получите разнообразие, извикайте API-то отново; за по-малко разнообразие, направете подсказката си по-конкретна.

## Допълнителни възможности за генериране на изображения

Вече видяхте как да генерирате изображение с няколко реда Python. Моделите `gpt-image` могат също да **редактират** съществуващо изображение. Като предоставите изображение, опционална **маска** (която маркира зоната за промяна) и подсказка, можете да промените част от изображение — например да добавите шапка на нашето зайче.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# промените също се връщат като base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Основното изображение съдържа само заека; крайното изображение има шапка на заека.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
